In [0]:
%pip install openpyxl

## Imports

In [0]:
from datetime import date, datetime
from pyspark.sql.functions import lit, format_number, coalesce, split, current_timestamp
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils
from io import BytesIO
import pandas as pd
import boto3
import json
import os

In [0]:
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        if aws_secret_key is not None:
            print("AWS secret key retrieved successfully.")
        else:
            print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix

## Get the needed Perimeter and Parameters

In [0]:
try:    
    report_args = dbutils.widgets.get("report_args")
    report_args = report_args.replace("\'", "\"")
except Exception as e:
    raise ValueError("Missing required input: report_args")

# Read JSON
json_data = json.loads(report_args)
 
# Access the values and store them into variables
if "output_file_prefix" not in json_data: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in json_data: raise KeyError("'report_output' key is missing in the report_args JSON data.")

output_file_prefix = "".join(json_data['output_file_prefix'])
report_output = json_data['report_output']

In [0]:
env = get_env_prefix()

In [0]:
# --- Execution mode: "portfolio" (original) or "perimeter" (offboarding/file-based) ---
try:
    execution_mode = dbutils.widgets.get("execution_mode")
except:
    execution_mode = "portfolio"
if not execution_mode:
    execution_mode = "portfolio"
if execution_mode not in ("portfolio", "perimeter"):
    raise ValueError("execution_mode must be 'portfolio' or 'perimeter'")

portfolioId = dbutils.widgets.get("portfolioId")
if execution_mode == "portfolio" and not portfolioId:
    raise ValueError("Missing required input: portfolioId")

run_date = dbutils.widgets.get("run_date")

run_date = spark.sql(f"""                    
                        select 
                            case 
                                when try_cast('{run_date}' as date) is not null then try_cast('{run_date}' as date)
                                else PreviousBusinessDay
                            end as BusinessDate
                        from `{env}-cdh`.reference.ref_calendar
                        where CalendarDate = coalesce(try_cast('{run_date}' as date),cast(getdate() as date))
                    """).first()[0]

print(f"Execution mode: {execution_mode} | Run date: {run_date}")

## Read Source Tables

In [0]:
dexposure = spark.sql(f"""
                            select distinct
                                      Exposurekey
                                    , Agreementkey
                                    , AssetSegment
                                    , PortfolioName
                                    , TotalPrincipal
                                    , TotalInterestAccruals
                                    , TotalExpense
                                    , OutstandingTotalBalance
                                    , BusinessDate
                                    , Currency
                                    , ExposureStatus
                                    , IsActiveFlag
                                    , TotalInterestL3869
                            from `{env}-cdh`.derivative.dexposure dex
                            where BusinessDate = '{run_date}'
                        """)

dexposure.createOrReplaceTempView("dexposure")

In [0]:
InitialExposure = spark.sql(f"""
                                select distinct
                                          Exposurekey
                                        , CDI
                                        , PartyRelationshipType
                                        , CaseCode
                                        , DiasNumber
                                        , SourceSystem
                                        , BusinessDate
                                from `{env}-cdh`.primary.exposure ex
                            """)

InitialExposure.createOrReplaceTempView("InitialExposure")

In [0]:
CaseCode = spark.sql(f"""                    
                        SELECT ExposureKey, CDI, CaseCode
                        FROM (
                            SELECT ExposureKey, CaseCode, CDI,
                                    ROW_NUMBER() OVER (PARTITION BY ExposureKey, CDI ORDER BY BusinessDate DESC) AS rn
                            FROM InitialExposure
                            WHERE SourceSystem in ('qc', 'qca')
                            and CaseCode is not null
                        ) t
                        WHERE rn = 1
                    """)

CaseCode.createOrReplaceTempView("CaseCode")

In [0]:
exposure = spark.sql(f"""
                            select distinct
                                      ex.Exposurekey
                                    , dex.Agreementkey
                                    , ex.CDI
                                    , dex.PortfolioName
                                    , ex.PartyRelationshipType
                                    , cc.CaseCode
                                    , ex.DiasNumber
                                    , ex.SourceSystem
                            from InitialExposure ex
                            inner join dexposure dex
                            on dex.ExposureKey = ex.ExposureKey
                            and ex.BusinessDate = dex.BusinessDate
                            left join CaseCode cc
                            on cc.CDI = ex.CDI
                            and (ex.ExposureKey = cc.ExposureKey or substring(ex.ExposureKey, 4, 50) = cc.ExposureKey)
                        """)

exposure.createOrReplaceTempView("exposure")

In [0]:
dparty = spark.sql(f"""
                            select distinct
                                      dp.CDI
                                    , PartyType
                                    , LegalNameGR
                                    , TaxRegistrationNumber
                                    , DeceasedFlag
                                    , NationalNumber
                                    , FirstFatherNameGR
                                    , PartykeyProfits
                            from `{env}-cdh`.derivative.dparty dp
                            where BusinessDate = '{run_date}'
                        """)

dparty.createOrReplaceTempView("dparty")

In [0]:
if execution_mode == "portfolio":
    portfolio_filter = f"and PortfolioId in ({','.join(portfolioId.split(','))})"
else:
    portfolio_filter = ""

Portfolio = spark.sql(f"""
                            select distinct
                                      PortfolioId
                                    , PortfolioName
                                    , Prefix
                            from `{env}-cdh`.primary.Portfolio
                            where BusinessDate = '{run_date}'
                            {portfolio_filter}
                        """)

Portfolio.createOrReplaceTempView("Portfolio")

In [0]:
masterXslave = spark.sql(f"""
                            select distinct
                                      EntityKey
                                    , RelatedEntityKey
                            from `{env}-cdh`.derivativeactive.dentityrelationships
                            where BusinessDate = '{run_date}'
                            and EntityRelationshipType = 'Master - Slave Rltnp'
                        """)

masterXslave.createOrReplaceTempView("masterXslave")

In [0]:
master = spark.sql(f"""
                        select distinct EntityKey MASTER_ACC
                        from masterXslave
                    """)

master.createOrReplaceTempView("master")

In [0]:
slave = spark.sql(f"""
                        select distinct RelatedEntityKey SLAVE_ACC
                        from masterXslave
                    """)

slave.createOrReplaceTempView("slave")

In [0]:
dagreement = spark.sql(f"""
                        select distinct
                                  Agreementkey
                                , AgreementStatus
                                , PortfolioName
                        from `{env}-cdh`.derivative.dagreement
                        where BusinessDate = '{run_date}'
                    """)

dagreement.createOrReplaceTempView("dagreement")

In [0]:
dunderlaw = spark.sql(f"""
                        select distinct
                                  CDI
                                , InLawFlag
                                , LawStatus
                        from `{env}-cdh`.derivative.dunderlaw
                        where BusinessDate = '{run_date}'
                        and UnderLawType = 'N3869'
                    """)

dunderlaw.createOrReplaceTempView("dunderlaw")

In [0]:
ContactDetails = spark.sql(f"""
                            select distinct
                                      CDI 
                                    , Street
                                    , PostalCode
                                    , City
                                    , Country
                                    , CommunicationFlag
                                    , ContactDetailType
                            from `{env}-cdh`.primary.ContactDetails
                            where BusinessDate = '{run_date}'
                            and Sourcesystem = 'profits'
                        """)

ContactDetails.createOrReplaceTempView("ContactDetails")

In [0]:
# Load perimeter only in perimeter mode
# Supports: (a) reference table, (b) Excel/CSV file from volume
if execution_mode == "perimeter":
    try:
        perimeter_file = dbutils.widgets.get("perimeter_file") or ""
    except:
        perimeter_file = ""
    
    if perimeter_file:
        # File-based perimeter from volume — only .xlsx accepted
        if not perimeter_file.endswith('.xlsx'):
            raise ValueError(f"Το perimeter αρχείο πρέπει να είναι .xlsx. Δόθηκε: '{perimeter_file}'")
        perimeter_path = f"/Volumes/p-cdh-delivery/internalreports/helloletter/input/{perimeter_file}"
        import pandas as pd
        pandas_df = pd.read_excel(perimeter_path)
        perimeter_df = spark.createDataFrame(pandas_df.astype(str))
        
        from pyspark.sql.functions import col
        cols = perimeter_df.columns
        cdi_col = next((c for c in ['CDI', 'cdi', 'CIF', 'cif'] if c in cols), None)
        ek_col = next((c for c in ['ExposureKey', 'Exposurekey', 'exposurekey'] if c in cols), None)
        
        if cdi_col and ek_col:
            offboarding_perimeter = perimeter_df.select(
                col(cdi_col).cast("string").alias("cdi"),
                col(ek_col).cast("string").alias("exposurekey")
            ).distinct()
        else:
            raise ValueError(f"Perimeter file must have CDI and ExposureKey columns. Found: {cols}")
        print(f"Perimeter loaded from file: '{perimeter_file}'")
    else:
        # Reference table perimeter
        try:
            perimeter_table = dbutils.widgets.get("perimeter_table")
        except:
            perimeter_table = ""
        if not perimeter_table:
            raise ValueError("Missing required input: perimeter_table (required when perimeter_file is not specified)")
        offboarding_perimeter = spark.sql(f"""
            SELECT DISTINCT cdi, exposurekey
            FROM {perimeter_table}
        """)
        print(f"Perimeter loaded from table: {perimeter_table}")
    
    offboarding_perimeter.createOrReplaceTempView("offboarding_perimeter")
    print(f"Offboarding perimeter: {offboarding_perimeter.count()} records")
else:
    print("Portfolio mode — no perimeter loaded")

## Create base tables

In [0]:
perimeter_join = "inner join offboarding_perimeter op on op.cdi = dp.CDI and op.exposurekey = dex.Exposurekey" if execution_mode == "perimeter" else ""

base = spark.sql(f"""
                    select *
                    from (
                        select distinct
                                row_number() over (partition by dp.CDI
                                                                , dex.Exposurekey 
                                                                order by case 
                                                                            when ex.PartyRelationshipType like 'Primary%' then 1 
                                                                            when ex.PartyRelationshipType like 'Guarantor%' then 2 
                                                                            else 3 
                                                                        end) rankrelation
                                , case 
                                    when ex.CaseCode is not null then 'Yes'
                                    else 'No'
                                end `Has Active CaseCode - CaseReferenceCodes`
                                , dp.CDI
                                , dp.taxregistrationnumber `ΑΦΜ`
                                , dp.NationalNumber `ΑΔΤ`
                                , dp.LegalNameGR `Εκτυπώσιμη Επωνυμία`
                                , case 
                                    when dp.PartyType = 'Individual' then dp.FirstFatherNameGR 
                                    else NULL 
                                end as `Πατρώνυμο`
                                , dp.deceasedflag DeceasedFlag
                                , substring(dex.Agreementkey, 4, 50) ContractNumber
                                , substring(dex.Exposurekey, 4, 50) Account
                                ,  ex.DiasNumber AS `ΔΙΑΣ PAYMENT`
                                , case dex.AssetSegment
                                    when 'CL' then 'Καταναλωτικό' 
                                    when 'ML' then 'Στεγαστικό' 
                                    else 'Επιχειρηματικό'
                                end `Product Type`
                                , dp.partytype PartyType
                                , p.PortfolioId
                                , p.PortfolioName
                                , concat('Agreement-', ex.PartyRelationshipType) AssetXPartyType
                                , concat('Loan-', ex.PartyRelationshipType) RelationshipType
                                , ex.CaseCode `QC Case Code`
                                , `{env}-cdh`.derivative.converttoeur(dex.Currency, dex.TotalPrincipal, '{run_date}') `Κεφάλαιο (€)`
                                , `{env}-cdh`.derivative.converttoeur(dex.Currency, dex.TotalInterestAccruals, '{run_date}') `Τόκοι (€)`
                                , `{env}-cdh`.derivative.converttoeur(dex.Currency, dex.TotalExpense, '{run_date}') `Έξοδα (€)`
                                , `{env}-cdh`.derivative.converttoeur(dex.Currency, dex.OutstandingTotalBalance, '{run_date}') `Σύνολο Οφειλής (€)`
                                , dex.ExposureStatus AccountStatus
                                , dp.partytype LegalEntity
                                , case
                                    when dagr.Agreementkey is null or AgreementStatus = 'Closed' then 'Inactive'
                                    else 'Active'
                                end `Agreement Status (Active/Inactive)`
                                , case 
                                    when s.SLAVE_ACC is not NULL then '0'
                                    when m.MASTER_ACC is not NULL then '1'
                                    else '-'
                                end as IsMasterFlag
                                , du.InLawFlag N3869Flag
                                , du.LawStatus N3869Status
                                , dex.TotalInterestL3869 `Ποσό απο τόκους υπο αναστολή στο πλαίσιο του Ν3869 (€)`
                        from dparty dp
                        inner join exposure ex
                            on dp.CDI = ex.CDI
                        inner join dexposure dex
                            on dex.Exposurekey = ex.Exposurekey
                        inner join Portfolio p
                            on p.PortfolioName = dex.PortfolioName
                        left join dagreement dagr
                        on dagr.Agreementkey = dex.Agreementkey
                        left join master m
                            on dex.Exposurekey = m.MASTER_ACC
                        left join slave s
                            on dex.Exposurekey = s.SLAVE_ACC
                        left join dunderlaw du
                            on du.CDI = dp.CDI
                        {perimeter_join}
                    )  as a 
                    where a.rankrelation = 1 
                """)

base.createOrReplaceTempView("base")

In [0]:
baserank = spark.sql(f"""
                        select distinct
                                  CDI
                                , `ΑΦΜ`
                                , `ΑΔΤ`
                                , `Εκτυπώσιμη Επωνυμία`
                                , `Πατρώνυμο`
                                , DeceasedFlag
                                , ContractNumber
                                , Account
                                , `ΔΙΑΣ PAYMENT`
                                , `Product Type`
                                , PartyType
                                , PortfolioName
                                , PortfolioId
                                , AssetXPartyType
                                , RelationshipType
                                , `QC Case Code`
                                , REPLACE(REPLACE(REPLACE(FORMAT_NUMBER(`Κεφάλαιο (€)`, 2), ',', 'X'), '.', ','), 'X', '.') AS `Κεφάλαιο (€)`
                                , REPLACE(REPLACE(REPLACE(FORMAT_NUMBER(`Τόκοι (€)`, 2), ',', 'X'), '.', ','), 'X', '.') AS `Τόκοι (€)`
                                , REPLACE(REPLACE(REPLACE(FORMAT_NUMBER(`Έξοδα (€)`, 2), ',', 'X'), '.', ','), 'X', '.') AS `Έξοδα (€)`
                                , REPLACE(REPLACE(REPLACE(FORMAT_NUMBER(`Σύνολο Οφειλής (€)`, 2), ',', 'X'), '.', ','), 'X', '.') AS `Σύνολο Οφειλής (€)`
                                , `AccountStatus`
                                , `LegalEntity`
                                , `Agreement Status (Active/Incative)`
                                , `IsMasterFlag`
                                , N3869Flag		
                                , N3869Status	
                                , `Ποσό απο τόκους υπο αναστολή στο πλαίσιο του Ν3869 (€)`
                                , Row_number() over (partition by CDI order by contractnumber, Account) rankd 
                        from base
                        {"where IsMasterFlag != '0' and `Agreement Status (Active/Incative)` != 'Inactive'" if execution_mode == "perimeter" else ""}
                    """)

baserank.createOrReplaceTempView("baserank")
baserank_pd = baserank.toPandas()

In [0]:
CDI = spark.sql(f"""
                    select distinct
                              CDI
                            , `ΑΦΜ`
                            , `ΑΔΤ`
                            , `Εκτυπώσιμη Επωνυμία`
                            , `Πατρώνυμο`
                            , DeceasedFlag
                            , PartyType
                            , N3869Flag		
                            , N3869Status
                            , PortfolioName
                    from baserank
                """)

CDI.createOrReplaceTempView("CDI")

In [0]:
max_rank = spark.sql(f"""
                    select MAX(rankd)
                    from baserank
                """).collect()[0][0]

In [0]:
# Define the fixed columns (they will NOT be pivoted)
Fixed_columns = [
      "CDI"
    , "ΑΦΜ"
    , "ΑΔΤ"
    , "Εκτυπώσιμη Επωνυμία"
    , "Πατρώνυμο"
    , "DeceasedFlag"
    , "N3869Flag"
    , "N3869Status"
]

# Define the pivotable columns (they will be pivoted)
Pivoted_columns = [
      "Product Type"
    , "ContractNumber"
    , "Account"
    , "IsMasterFlag"
    , "ΔΙΑΣ PAYMENT"
    , "Κεφάλαιο (€)"
    , "Τόκοι (€)"
    , "Έξοδα (€)"
    , "Σύνολο Οφειλής (€)"
    , "Ποσό απο τόκους υπο αναστολή στο πλαίσιο του Ν3869 (€)"
]

# Define the mapping for renaming columns
rename_mapping = {
    "ContractNumber": "APPLICATION NUMBER",
    "Account": "ACCOUNT NUMBER",
    "Τόκοι (€)": "Τόκοι (€) επί Ενήμερου & Ληξ/μου Κεφαλαίου (συμπ οι αναλογούντες  δεδουλευμένοι τόκοι)",
}

In [0]:
# Fixed columns
df_fixed = baserank_pd[Fixed_columns].drop_duplicates()

# Pivot each column in the Pivoted_columns list
df_pivoted = pd.DataFrame()

for col in Pivoted_columns:
    df_temp = baserank_pd.pivot(index='CDI', columns='rankd', values=col)
    new_col_name = rename_mapping.get(col, col)
    df_temp.columns = [f'{new_col_name} {i}' for i in df_temp.columns]
    df_temp.reset_index(inplace=True)
    df_pivoted = pd.merge(df_pivoted, df_temp, on='CDI', how='outer') if not df_pivoted.empty else df_temp

# Fixed + Pivoted columns
CDIContract_part1_pd = pd.merge(df_pivoted, df_fixed, on='CDI')

# Reorder the columns in the final DataFrame
pivoted_columns_order = [f'{rename_mapping.get(col, col)} {i}' for i in range(1, max_rank + 1) for col in Pivoted_columns]
final_column_order = Fixed_columns + pivoted_columns_order
CDIContract_part1_pd = CDIContract_part1_pd[final_column_order]

# Create the final DataFrames
CDIContract_pivoted_pd = CDIContract_part1_pd[['CDI'] + pivoted_columns_order].rename(columns={'CDI': 'CDI_pivoted'})
CDIContract_pivoted = spark.createDataFrame(CDIContract_pivoted_pd)
CDIContract_pivoted.createOrReplaceTempView("CDIContract_pivoted")

CDIContract_part1 = spark.createDataFrame(CDIContract_part1_pd)
CDIContract_part1.createOrReplaceTempView("CDIContract_part1")

In [0]:
Address = spark.sql(f"""
                        select *
                        from (
                            select distinct
                                      d.CDI
                                    , Street `Εκτυπώσιμη διεύθυνση`
                                    , PostalCode `ΤΚ`
                                    , City `Περιοχή - Πόλη`
                                    , Country `Χώρα`
                                    , rank() over (partition by d.CDI 
                                                order by CommunicationFlag desc
                                                        , case 
                                                            when ContactDetailType = 'Head Office' then 1 
                                                            when ContactDetailType = 'Work Address' then 2 
                                                            else 3 
                                                            end
                                                        ) rnk
                            from baserank d
                            inner join ContactDetails c
                            on d.CDI = c.CDI
                        ) as a
                        where rnk = 1
                    """)

Address.createOrReplaceTempView("Address")

## Horizontal File

In [0]:
Horizontal_file = spark.sql(f"""
                                select distinct
                                          p.PortfolioName
                                        , c.CDI AS CIF
                                        , c.`ΑΦΜ`
                                        , c.`ΑΔΤ`
                                        , c.`Εκτυπώσιμη Επωνυμία`
                                        , c.`Πατρώνυμο`
                                        , c.DeceasedFlag
                                        , c.N3869Flag			
                                        , c.N3869Status			
                                        , c.PartyType
                                        , a.`Εκτυπώσιμη διεύθυνση`
                                        , a.`ΤΚ`
                                        , a.`Περιοχή - Πόλη`
                                        , a.`Χώρα`
                                        , CDIContract_pivoted.*
                                from CDI c
                                inner join Portfolio p
                                    on substring(c.CDI, 1, 3) = p.Prefix
                                inner join Address a
                                    on c.CDI = a.CDI
                                inner join CDIContract_pivoted
                                    on CDIContract_pivoted.CDI_pivoted = c.CDI
                            """)
                    
Horizontal_file = Horizontal_file.drop('CDI_pivoted')
Horizontal_file.createOrReplaceTempView("Horizontal_file")

## Vertical File

In [0]:
Vertical_file = spark.sql(f"""
                              select distinct
                                    b.CDI
                                  , b.Account
                                  , b.PortfolioId
                                  , b.ContractNumber
                                  , b.`QC Case Code`
                                  , b.RelationshipType
                                  , b.`AccountStatus`
                                  , `Σύνολο Οφειλής (€)` `TotalLegalClaim (€)`
                                  , b.`LegalEntity`
                                  , b.IsMasterFlag	
                                  , b.N3869Flag	
                                  , b.N3869Status	
                                  , case 
                                      when b.`Ποσό απο τόκους υπο αναστολή στο πλαίσιο του Ν3869 (€)` is null then 'no'
                                      else 'yes'
                                    end as has_tokoi_flag
                                  , b.`Ποσό απο τόκους υπο αναστολή στο πλαίσιο του Ν3869 (€)` `Ποσό απο τόκους υπο αναστολή στο πλαίσιο του Ν3869`
                              from baserank b
                          """)
                    
Vertical_file.createOrReplaceTempView("Vertical_file")

## Exclusion File

In [0]:
ContractsWithZeroTotalBalance = spark.sql(f"""
                                            select distinct ContractNumber
                                            from (
                                                select distinct
                                                        ContractNumber
                                                        , sum(`Σύνολο Οφειλής (€)`) TotalBalance
                                                from base
                                                group by all
                                            ) SumOfTotalBalanceForContracts
                                            where TotalBalance = 0
                                        """)
                    
ContractsWithZeroTotalBalance.createOrReplaceTempView("ContractsWithZeroTotalBalance")

In [0]:
perimeter_ex_join = "inner join offboarding_perimeter op on op.cdi = ex.CDI and op.exposurekey = ex.Exposurekey" if execution_mode == "perimeter" else ""

ExclusionsContractsWithZeroTotalBalance = spark.sql(f"""
                                                        select distinct
                                                                  ex.CDI
                                                                , cwztb.ContractNumber
                                                                , ex.Exposurekey Account
                                                                , p.PortfolioId
                                                                , concat('Loan-', ex.PartyRelationshipType, ' Rltp') RelationshipType
                                                                , 'Agreement has zero total balance' ExclusionReason
                                                        from ContractsWithZeroTotalBalance cwztb
                                                        inner join exposure ex
                                                        on substring(ex.Agreementkey, 4, 50) = cwztb.ContractNumber
                                                        inner join Portfolio p
                                                        on p.PortfolioName = ex.PortfolioName
                                                        {perimeter_ex_join}
                                                    """)
                    
ExclusionsContractsWithZeroTotalBalance.createOrReplaceTempView("ExclusionsContractsWithZeroTotalBalance")

In [0]:
# In perimeter mode baserank already excludes Inactive rows, so query base directly
inactive_source = "base" if execution_mode == "perimeter" else "baserank"
baserankInactiveContracts = spark.sql(f"""
                                            select distinct ContractNumber
                                            from {inactive_source}
                                            where `Agreement Status (Active/Incative)` = 'Inactive'
                                        """)
                    
baserankInactiveContracts.createOrReplaceTempView("baserankInactiveContracts")

In [0]:
ExclusionsInactiveContracts = spark.sql(f"""
                                            select distinct
                                                      ex.CDI
                                                    , bic.ContractNumber
                                                    , ex.Exposurekey Account
                                                    , p.PortfolioId
                                                    , concat('Loan-', ex.PartyRelationshipType, ' Rltp') RelationshipType
                                                    , 'Agreement is closed' ExclusionReason
                                            from baserankInactiveContracts bic
                                            inner join exposure ex
                                            on substring(ex.Agreementkey, 4, 50) = bic.ContractNumber
                                            inner join Portfolio p
                                            on p.PortfolioName = ex.PortfolioName
                                            {perimeter_ex_join}
                                        """)
                    
ExclusionsInactiveContracts.createOrReplaceTempView("ExclusionsInactiveContracts")

In [0]:
perimeter_slave_join = "inner join offboarding_perimeter op on op.exposurekey = slave_dex.Exposurekey" if execution_mode == "perimeter" else ""

AgreementsOnlySlaveAccounts = spark.sql(f"""
                                            select distinct slave_dex.Agreementkey ContractNumber
                                            from masterXslave
                                            inner join dexposure master_dex
                                            on master_dex.Exposurekey = masterXslave.EntityKey
                                            inner join dexposure slave_dex
                                            on slave_dex.Exposurekey = masterXslave.RelatedEntityKey
                                            inner join Portfolio p
                                            on p.PortfolioName = master_dex.PortfolioName
                                            and p.PortfolioName = slave_dex.PortfolioName
                                            {perimeter_slave_join}
                                            where master_dex.Agreementkey <> slave_dex.Agreementkey
                                        """)
                   
AgreementsOnlySlaveAccounts.createOrReplaceTempView("AgreementsOnlySlaveAccounts")

In [0]:
ExclusionsAgreementOnlySlaveAccounts = spark.sql(f"""
                                                    select distinct
                                                              ex.CDI
                                                            , osa.ContractNumber
                                                            , ex.Exposurekey Account
                                                            , p.PortfolioId
                                                            , concat('Loan-', ex.PartyRelationshipType, ' Rltp') RelationshipType
                                                            , 'Agreement contains only slave accounts' ExclusionReason
                                                    from AgreementsOnlySlaveAccounts osa
                                                    inner join exposure ex
                                                    on ex.Agreementkey = osa.ContractNumber
                                                    inner join Portfolio p
                                                    on p.PortfolioName = ex.PortfolioName
                                                    {perimeter_ex_join}
                                                """)
                    
ExclusionsAgreementOnlySlaveAccounts.createOrReplaceTempView("ExclusionsAgreementOnlySlaveAccounts")

In [0]:
if execution_mode == "perimeter":
    # Contracts without loans cannot be in the perimeter (perimeter is CDI+ExposureKey based)
    ContractsWithoutLoans = spark.createDataFrame([], "ContractNumber STRING")
else:
    ContractsWithoutLoans = spark.sql(f"""
                                        select distinct dagr.Agreementkey ContractNumber
                                        from dagreement dagr
                                        inner join Portfolio p
                                            on p.PortfolioName = dagr.PortfolioName
                                        left join dexposure dex
                                            on dagr.Agreementkey = dex.Agreementkey
                                        where dex.Exposurekey is null
                                    """)

ContractsWithoutLoans.createOrReplaceTempView("ContractsWithoutLoans")

In [0]:
ExclusionsContractsWithoutLoans = spark.sql(f"""
                                                select distinct
                                                          NULL CDI
                                                        , cwl.ContractNumber
                                                        , NULL Account
                                                        , p.PortfolioId
                                                        , NULL RelationshipType
                                                        , 'Agreement without loans' ExclusionReason
                                                from ContractsWithoutLoans cwl
                                                inner join Portfolio p
                                                on p.Prefix = left(cwl.ContractNumber,3)
                                            """)
                    
ExclusionsContractsWithoutLoans.createOrReplaceTempView("ExclusionsContractsWithoutLoans")

In [0]:
Exclusion_file = ExclusionsContractsWithZeroTotalBalance.union(ExclusionsAgreementOnlySlaveAccounts) \
                                                        .union(ExclusionsContractsWithoutLoans) \
                                                        .union(ExclusionsInactiveContracts)

## Export to Volume

In [0]:
# notebook: {env}-cdh-delivery
# volume: {env}-cdh-delivery.internalreports.helloletter
# export path: \\dwhsideapps\Share\in\{env}\InternalReports\helloletter

In [0]:
%pip install openpyxl

In [0]:
from pyspark.sql.functions import col

# Define the output directory path
run_date_str = datetime.now().strftime("%Y%m%d")
output_dir = f"{report_output}{run_date_str}/"

# Ensure the directory exists
os.makedirs(output_dir, exist_ok=True)

def export_to_excel(df, sheet_name):
    
    # Convert columns with dtype 'void' to string
    for col_name, dtype in df.dtypes:
        if dtype == 'void':
            df = df.withColumn(col_name, col(col_name).cast('string'))

    # Define the output file path
    output_filename = f"{output_file_prefix}_{sheet_name}_{run_date.strftime('%Y%m%d')}.xlsx"
    output_path = os.path.join(output_dir, output_filename)

    # Create an Excel writer object
    output_buffer = BytesIO()
    with pd.ExcelWriter(output_buffer, engine='openpyxl') as writer:
        # Save the subset to a temporary CSV file
        temp_file = os.path.join(output_dir, f"Hello_Letter.csv")
        df.coalesce(1).write.csv(temp_file, header=True, mode="overwrite")
        
        # Read the CSV back into Pandas
        csv_files = [f for f in os.listdir(temp_file) if f.endswith(".csv")]
        if csv_files:
            pandas_df = pd.read_csv(os.path.join(temp_file, csv_files[0]), dtype=str)
            
            # Write to an Excel sheet
            pandas_df.to_excel(writer, sheet_name=sheet_name, index=False)
        for file in os.listdir(temp_file):
            os.remove(os.path.join(temp_file, file))

    with open(output_path, "wb") as f:
        f.write(output_buffer.getvalue())

In [0]:
export_to_excel(Horizontal_file, 'Horizontal')
export_to_excel(Vertical_file, 'Vertical')
export_to_excel(Exclusion_file, 'Exclusion')